In [0]:
BASE = "/Volumes/learning/bronze/raw"

orders_raw = (
    spark.read
        .option("header", True)
       # .option("inferSchema", True)
        .csv(f"{BASE}/orders")
)

orders_raw.printSchema()
display(orders_raw.limit(10))


root
 |-- order_id: integer (nullable = true)
 |-- customer_id: integer (nullable = true)
 |-- product_id: integer (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- unit_price: double (nullable = true)
 |-- order_timestamp: string (nullable = true)
 |-- status: string (nullable = true)



order_id,customer_id,product_id,quantity,unit_price,order_timestamp,status
1001,178,28,4,196.05,2026-10-01 05:21:26,shipped
1002,141,22,1,237.41,2026-10-01 12:52:48,cancelled
1003,30,3,4,184.84,2026-10-01 10:16:20,completed
1004,14,22,2,237.41,2026-10-01 17:29:26,shipped
1005,195,20,1,207.78,2026-10-01 15:40:28,pending
1006,74,9,4,51.33,2026-10-01 17:08:59,cancelled
1007,62,4,2,223.33,2026-10-01 00:40:38,shipped
1008,24,18,2,212.38,2026-10-01 00:35:26,completed
1009,40,15,4,90.92,2026-10-01 03:41:53,shipped
1010,124,17,4,213.55,2026-10-01 22:17:26,pending


In [0]:
orders_schema = """
    order_id    INT,
    customer_id INT,
    product_id  INT,
    quantity    INT,
    unit_price  DOUBLE,
    order_timestamp TIMESTAMP,
    status      STRING
"""

orders = (
    spark.read
        .schema(orders_schema)
        .option("header", True)
        .csv(f"{BASE}/orders")
)

orders.printSchema()
display(orders)
#

root
 |-- order_id: integer (nullable = true)
 |-- customer_id: integer (nullable = true)
 |-- product_id: integer (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- unit_price: double (nullable = true)
 |-- order_timestamp: timestamp (nullable = true)
 |-- status: string (nullable = true)



order_id,customer_id,product_id,quantity,unit_price,order_timestamp,status
1001,178,28,4,196.05,2026-10-01T05:21:26.000Z,shipped
1002,141,22,1,237.41,2026-10-01T12:52:48.000Z,cancelled
1003,30,3,4,184.84,2026-10-01T10:16:20.000Z,completed
1004,14,22,2,237.41,2026-10-01T17:29:26.000Z,shipped
1005,195,20,1,207.78,2026-10-01T15:40:28.000Z,pending
1006,74,9,4,51.33,2026-10-01T17:08:59.000Z,cancelled
1007,62,4,2,223.33,2026-10-01T00:40:38.000Z,shipped
1008,24,18,2,212.38,2026-10-01T00:35:26.000Z,completed
1009,40,15,4,90.92,2026-10-01T03:41:53.000Z,shipped
1010,124,17,4,213.55,2026-10-01T22:17:26.000Z,pending


In [0]:
big_orders = orders.filter(orders.quantity >= 4)
#print("Defined big_orders. Nothing has been computed yet!")
display(big_orders.limit(10000))
print("Displayed big_orders. It has been computed now!")


order_id,customer_id,product_id,quantity,unit_price,order_timestamp,status
1001,178,28,4,196.05,2026-10-01T05:21:26.000Z,shipped
1003,30,3,4,184.84,2026-10-01T10:16:20.000Z,completed
1006,74,9,4,51.33,2026-10-01T17:08:59.000Z,cancelled
1009,40,15,4,90.92,2026-10-01T03:41:53.000Z,shipped
1010,124,17,4,213.55,2026-10-01T22:17:26.000Z,pending
1011,null,15,5,90.92,2026-10-01T17:09:24.000Z,pending
1013,9999,27,5,165.65,2026-10-01T00:18:46.000Z,cancelled
1016,199,19,4,138.94,null,completed
1017,98,11,4,27.14,null,pending
1022,195,11,4,27.14,2026-10-01T20:45:56.000Z,shipped


Displayed big_orders. It has been computed now!


In [0]:
from pyspark.sql import functions as F

orders_enriched = (
    orders
        .select("order_id", "customer_id", "product_id", "quantity", "unit_price", "status")
        .filter(F.col("quantity") > 0)
        .withColumn("line_total",
            F.round(F.col("quantity") * F.col("unit_price"), 2))
        .withColumn("Status_clean",
            F.lower(F.trim(F.col("status"))))
)
display(orders_enriched)

order_id,customer_id,product_id,quantity,unit_price,status,line_total,Status_clean
1001,178,28,4,196.05,shipped,784.2,shipped
1002,141,22,1,237.41,cancelled,237.41,cancelled
1003,30,3,4,184.84,completed,739.36,completed
1004,14,22,2,237.41,shipped,474.82,shipped
1005,195,20,1,207.78,pending,207.78,pending
1006,74,9,4,51.33,cancelled,205.32,cancelled
1007,62,4,2,223.33,shipped,446.66,shipped
1008,24,18,2,212.38,completed,424.76,completed
1009,40,15,4,90.92,shipped,363.68,shipped
1010,124,17,4,213.55,pending,854.2,pending


In [0]:
print(big_orders.count())

212


In [0]:
%sql
SELECT order_id, quantity*unit_price AS line_total, lower(trim(status)) AS status_clean
FROM read_files('/Volumes/learning/bronze/raw/orders/', format => 'csv', header => true)
WHERE quantity > 0
LIMIT 10;

order_id,line_total,status_clean
1001,784.2,shipped
1002,237.41,cancelled
1003,739.36,completed
1004,474.82,shipped
1005,207.78,pending
1006,205.32,cancelled
1007,446.66,shipped
1008,424.76,completed
1009,363.68,shipped
1010,854.2,pending


In [0]:
display(
    orders.groupBy("quantity").count().orderBy("quantity")
)

quantity,count
-5,1
0,1
1,97
2,98
3,96
4,113
5,99


In [0]:
orders_schema_with_corrupt = orders_schema + ", _corrupt_record STRING"

orders_checked = (
    spark.read
        .schema(orders_schema_with_corrupt)
        .option("header", True)
        .option("mode", "PERMISSIVE")
        .option("columnNameOfCorruptRecord", "_corrupt_record")
        .csv(f"{BASE}/orders")
)
display(orders_checked)

order_id,customer_id,product_id,quantity,unit_price,order_timestamp,status,_corrupt_record
1001,178,28,4,196.05,2026-10-01T05:21:26.000Z,shipped,null
1002,141,22,1,237.41,2026-10-01T12:52:48.000Z,cancelled,null
1003,30,3,4,184.84,2026-10-01T10:16:20.000Z,completed,null
1004,14,22,2,237.41,2026-10-01T17:29:26.000Z,shipped,null
1005,195,20,1,207.78,2026-10-01T15:40:28.000Z,pending,null
1006,74,9,4,51.33,2026-10-01T17:08:59.000Z,cancelled,null
1007,62,4,2,223.33,2026-10-01T00:40:38.000Z,shipped,null
1008,24,18,2,212.38,2026-10-01T00:35:26.000Z,completed,null
1009,40,15,4,90.92,2026-10-01T03:41:53.000Z,shipped,null
1010,124,17,4,213.55,2026-10-01T22:17:26.000Z,pending,null


In [0]:
customers = (
    spark.read.option("header", True)
    .schema("customer_id INT, name STRING, country STRING, signedup_date DATE")
        .csv(f"{BASE}/customers")
)

products = (
    spark.read.option("header", "true")
    .schema("prouct_id INT, product_name STRING, category STRING, price DOUBLE ")
    .csv(f"{BASE}/products/")
)

print("customers: ", customers.count())
print("products: ", products.count())
display(customers)
display(products)

customers:  201
products:  30


customer_id,name,country,signedup_date
1,Anna Novak,Spain,2025-04-25
2,Sara Müller,France,2025-08-17
3,Anna Schmidt,France,2026-03-09
4,Jonas Jansen,Germany,2025-08-12
5,Jonas Jansen,Austria,2026-07-29
6,Paul Kaya,Italy,2025-08-14
7,Sara Dubois,Poland,2025-01-07
8,Sara Garcia,Poland,2025-10-12
9,Lukas Dubois,France,2025-04-15
10,Leon Kaya,Italy,2026-01-03


prouct_id,product_name,category,price
1,Headphones,Electronics,95.09
2,Keyboard,Electronics,188.2
3,Mouse,Electronics,184.84
4,Monitor,Electronics,-10.0
5,Webcam,Electronics,284.54
6,Speaker,Electronics,66.3
7,Lamp,Home,67.25
8,Chair,Home,199.83
9,Desk,Home,51.33
10,Pillow,Home,56.27


In [0]:
print("all rows:                   ", orders.count())
print("full-row duplicates removed:", orders.dropDuplicates().count())
print("unique order_ids:           ", orders.dropDuplicates(["order_id"]).count())

all rows:                    505
full-row duplicates removed: 500
unique order_ids:            500


In [0]:
customer_dedup = customers.dropDuplicates(["customer_id"])
print("customers: ", customers.count())
print("deduped customers: ", customer_dedup.count())

customers:  201
deduped customers:  200


In [0]:
with_dup    = orders_enriched.join(customers,   "customer_id", "inner").count()
without_dup = orders_enriched.join(customer_dedup, "customer_id", "inner").count()

print("join with duplicate customer:", with_dup)
print("join after dedup:           ", without_dup)

join with duplicate customer: 500
join after dedup:            500


In [0]:
display(orders.filter("customer_id = 6"))


order_id,customer_id,product_id,quantity,unit_price,order_timestamp,status


In [0]:
customers_doubled = customer_dedup.union(customers_dedup)